# Week 10: A complete reproducible ML workflow

Machine Learning in Engineering Science | NCKU + LAiMM | Fall 2026

**Purpose:** a fully worked, self-paced in-class practice with an optional extension. Weeks 11 and 18 are optional examination-readiness materials. These notebooks do not create weekly graded assignments.

**Before running:** choose a Python 3 kernel in Jupyter or a CPU runtime in Colab. Run every cell from top to bottom with Shift+Enter. Setup installs missing free packages. No API key or external dataset download is required.

**Learning target:** complete the calculation, inspect intermediate values, and explain why the protocol supports a limited claim.

**Time:** about 25-45 minutes including reading, prediction, execution, and explain-back.


## 1. Set up and record the environment

Keep the seed fixed for the first run. Record actual outputs and package versions rather than copying a claimed performance number.


In [ ]:
import sys, subprocess, importlib.util, platform, json, random, hashlib
from pathlib import Path
required = {"numpy": "numpy", "matplotlib": "matplotlib", "sklearn": "scikit-learn"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
import numpy as np
import matplotlib.pyplot as plt
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
print("Python:", platform.python_version(), "NumPy:", np.__version__, "Seed:", SEED)


**Checkpoint.** The environment cell must finish before any analysis cell. Restart and run all if a variable is missing.


## 2. Create a frozen split

Persist sample IDs as well as a seed. A seed alone is insufficient if input row order changes.


In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
data=load_diabetes(); X,y=data.data,data.target
train_ids,test_ids=train_test_split(np.arange(len(y)),test_size=.25,random_state=SEED)
print("Train/test counts:",len(train_ids),len(test_ids))


**Checkpoint.** This packaged dataset is used as a statistical teaching example, not for clinical decisions.


## 3. Fit and serialize evidence

Use a fixed hyperparameter for this exercise. A real selection exercise needs an inner validation split or cross-validation.


In [ ]:
model=make_pipeline(StandardScaler(),Ridge(alpha=20))
model.fit(X[train_ids],y[train_ids]); predictions=model.predict(X[test_ids])
summary={"test_MAE":float(mean_absolute_error(y[test_ids],predictions)),"alpha":20,
         "train_ids":train_ids.tolist(),"test_ids":test_ids.tolist()}
np.savez(ARTIFACTS/"workflow-data.npz",X=X,y=y,train_ids=train_ids,test_ids=test_ids)
print("MAE:",summary["test_MAE"])


**Checkpoint.** The test IDs and baseline configuration must be included in the report.


## 4. Reproduce in a fresh model object

Rebuild the same pipeline and fit on identical rows. Compare numerical predictions with an explicit tolerance.


In [ ]:
replica=make_pipeline(StandardScaler(),Ridge(alpha=20)).fit(X[train_ids],y[train_ids])
delta=float(np.max(np.abs(replica.predict(X[test_ids])-predictions)))
assert delta<1e-8
summary["rerun_max_delta"]=delta
print("Rerun maximum prediction difference:",delta)


**Checkpoint.** Passing this deterministic rerun is narrower than independent scientific replication.


## 6. Preserve your evidence

Export a JSON record with a seed, package version, and measured result. Download the artifacts directory before leaving Colab; runtime storage is temporary.


In [ ]:
record={"week":10,"seed":SEED,"numpy":np.__version__,"python":platform.python_version(),"result":summary}
(ARTIFACTS/"week-10-evidence.json").write_text(json.dumps(record,indent=2))
print(json.dumps(record,indent=2))


**Checkpoint.** The evidence file must match the result you actually obtained. A saved screenshot alone does not permit rerunning the calculation.


## 7. Try one change and explain it
1. Predict the effect of one change before editing code. Change one parameter, input, or boundary condition only.
2. Rerun all cells in a fresh kernel. Keep the original evidence record and compare the observed difference.
3. Explain the result using the mechanism, not just the final number.
4. Name one assumption that limits the claim, and a new test that could challenge it.

**Troubleshooting:** if installation fails, restore network access; if a variable is undefined, run earlier cells; if your output differs, compare package versions, seed, and data construction. Do not repeatedly edit code until a preferred score appears.

**Completion criterion:** one executed notebook, one evidence JSON, and a brief claim/evidence/limitation statement.

### Reference documentation
- [NumPy user guide](https://numpy.org/doc/stable/user/)
- [scikit-learn user guide](https://scikit-learn.org/stable/user_guide.html)
- [Course repository](https://github.com/CHjonnyYu/MLES-Fall-2026)
